In [17]:
import pandas as pd
import numpy as np

In [18]:
df = pd.read_csv('college_major_career_outcomes_2026.csv')
print(df.head())

            program_id    unitid  opeid6  \
0  opeid031505-51.07-1       NaN   31505   
1  opeid031505-51.35-1       NaN   31505   
2  opeid031505-52.04-1       NaN   31505   
3       493868-12.04-1  493868.0   42833   
4       177834-13.06-5  177834.0    2477   

                          institution_name  institution_control  \
0                    A - Technical College  Private, for-profit   
1                    A - Technical College  Private, for-profit   
2                    A - Technical College  Private, for-profit   
3         A Better U Beauty Barber Academy  Private, for-profit   
4  A T Still University of Health Sciences   Private, nonprofit   

   is_main_campus institution_city institution_state institution_region  \
0               1              NaN               NaN                NaN   
1               1              NaN               NaN                NaN   
2               1              NaN               NaN                NaN   
3               1      Albuquerq

In [19]:
print({"rows": int(df.shape[0]), "columns": int(df.shape[1])})

{'rows': 39999, 'columns': 72}


In [20]:
print(df.shape[0]), print(df.shape[1])

39999
72


(None, None)

In [21]:
print( {
        "unique_program_ids": int(df["program_id"].nunique()),
        "unique_rows_excluding_id": int(
            df.drop(columns=["program_id"]).astype(str).drop_duplicates().shape[0]
        ),
    })

{'unique_program_ids': 39999, 'unique_rows_excluding_id': 39999}


In [22]:
counts = df.isna().sum()
print( {column: int(count) for column, count in counts.items() if count > 0})

{'unitid': 1168, 'institution_city': 2123, 'institution_state': 2123, 'institution_region': 2123, 'institution_latitude': 2123, 'institution_longitude': 2123, 'institution_is_hbcu': 2123, 'institution_admission_rate': 19786, 'institution_avg_sat': 27311, 'institution_undergrad_enrollment': 2546, 'institution_tuition_in_state_usd': 4883, 'institution_tuition_out_state_usd': 4883, 'awards_year1': 5325, 'awards_year2': 5125, 'median_earnings_4yr_usd': 29912, 'earnings_cohort_size_4yr': 29912, 'national_median_earnings_4yr_usd': 12716, 'national_p25_earnings_4yr_usd': 12833, 'national_p75_earnings_4yr_usd': 12782, 'earnings_vs_national_pct': 29913, 'median_earnings_1yr_usd': 31084, 'earnings_cohort_size_1yr': 31084, 'median_earnings_5yr_usd': 31113, 'earnings_cohort_size_5yr': 31113, 'not_working_count_5yr': 16231, 'count_above_hs_threshold_5yr': 32373, 'count_working_in_state_5yr': 33004, 'median_debt_usd': 32189, 'debt_borrower_count': 30370, 'median_monthly_payment_usd': 32189, 'earning

In [23]:
columns = [
        "institution_name",
        "institution_control",
        "institution_city",
        "institution_state",
        "institution_region",
    ]
print({column: int(df[column].nunique(dropna=True)) for column in columns})

{'institution_name': 1439, 'institution_control': 4, 'institution_city': 741, 'institution_state': 55, 'institution_region': 10}


In [26]:
per_major = df.groupby("institution_name", observed=True)["institution_control"].nunique()
print({ "majors_with_multiple_career_outcomes": int((per_major > 1).sum()) })   

{'majors_with_multiple_career_outcomes': 3}


In [ ]:
CONTROL = "institution_control"

shares = df[CONTROL].value_counts(normalize=True) * 100
majority_share = float(shares.iloc[0]) if not shares.empty else 0.0

print( {
    **{
        str(control): round(float(share), 2)
        for control, share in shares.items()
    },
    "majority_institution_control_pct": round(majority_share, 2),
    "majority_baseline_accuracy_pct": round(ajority_share, 2),
})

{'Public': 55.04, 'Private, nonprofit': 35.7, 'Private, for-profit': 8.31, 'Foreign': 0.96, 'majority_institution_control_pct': 55.04, 'majority_baseline_accuracy_pct': 55.04}


In [29]:
CONTROL = "institution_control"
REGION = "institution_region"

control_counts = df[CONTROL].value_counts()
majority_control = control_counts.index[0] if not control_counts.empty else None

at_control = df.loc[df[CONTROL] == majority_control]
same_region = at_control.loc[at_control[REGION] == df[REGION].mode()[0]]

print( {
    "rows_at_majority_control": int(len(at_control)),
    "rows_in_majority_region": int(len(same_region)),
    "control_region_association": bool(len(at_control) == len(same_region)),
    "majority_region_at_other_controls": int(
        len(df.loc[
            (df[REGION] == df[REGION].mode()[0]) &
            (df[CONTROL] != majority_control)
        ])
    ),
})

{'rows_at_majority_control': 22015, 'rows_in_majority_region': 5242, 'control_region_association': False, 'majority_region_at_other_controls': 3521}


In [30]:
CONTROL = "institution_control"
REGION = "institution_region"

late = df[CONTROL].isin(df[CONTROL].value_counts().nlargest(2).index)

print( {
    "institutions_in_top_2_control_types": int(late.sum()),
    "institutions_in_top_2_control_types_and_top_region": int(
        (late & (df[REGION] == df[REGION].mode()[0])).sum()
    ),
})

{'institutions_in_top_2_control_types': 36294, 'institutions_in_top_2_control_types_and_top_region': 8111}


In [31]:
control_counts = df["institution_control"].value_counts()

public = df.loc[df["institution_control"].str.lower() == "public"]
private = df.loc[df["institution_control"].str.lower() == "private"]

print( {
    "total_institutions": int(df["institution_name"].nunique()),
    "public_institutions": int(len(public)),
    "private_institutions": int(len(private)),
    "public_institutions_pct": round(len(public) / len(df) * 100, 1),
    "private_institutions_pct": round(len(private) / len(df) * 100, 1),
    "public_unique_states": int(public["institution_state"].nunique()),
    "private_unique_states": int(private["institution_state"].nunique()),
    "public_unique_regions": int(public["institution_region"].nunique()),
    "private_unique_regions": int(private["institution_region"].nunique()),
})

{'total_institutions': 1439, 'public_institutions': 22015, 'private_institutions': 0, 'public_institutions_pct': 55.0, 'private_institutions_pct': 0.0, 'public_unique_states': 48, 'private_unique_states': 0, 'public_unique_regions': 10, 'private_unique_regions': 0}


In [39]:
print (
        df.groupby("institution_control", observed=True)["institution_name"]
          .nunique()
          .div(df["institution_name"].nunique())
          .mul(100)
    )

institution_control
Foreign                 2.849201
Private, for-profit    43.293954
Private, nonprofit     29.256428
Public                 24.878388
Name: institution_name, dtype: float64


In [45]:
rates = (
        df["institution_region"]
        .value_counts(normalize=True)
        .mul(100)
        .sort_values()
    )

print( {
        "column": "institution_region",
        "levels": int(len(rates)),
        "min_pct": round(float(rates.iloc[0]), 2),
        "max_pct": round(float(rates.iloc[-1]), 2),
        "spread_pp": round(float(rates.iloc[-1] - rates.iloc[0]), 2),
        "lowest_level": str(rates.index[0]),
        "highest_level": str(rates.index[-1]),
    })

{'column': 'institution_region', 'levels': 10, 'min_pct': 0.12, 'max_pct': 23.14, 'spread_pp': 23.02, 'lowest_level': 'U.S. Service Schools', 'highest_level': 'Southeast'}


In [46]:

rates = (
        df["institution_region"]
        .value_counts(normalize=True)
        .mul(100)
    )

present = list(rates.index)

result = {
        str(region): round(float(rates[region]), 2)
        for region in present
    }

result["spread_pp"] = round(
        max(rates.values) - min(rates.values), 2
    )

print(result)

{'Southeast': 23.14, 'Far West': 19.78, 'Mid East': 15.48, 'Great Lakes': 12.84, 'Southwest': 8.48, 'Plains': 6.79, 'New England': 6.58, 'Rocky Mountains': 5.72, 'Outlying Areas': 1.09, 'U.S. Service Schools': 0.12, 'spread_pp': np.float64(23.02)}


In [47]:
rates = df["institution_region"].value_counts(normalize=True) * 100

print( {
        "highest_region_pct": round(float(rates.max()), 2),
        "lowest_region_pct": round(float(rates.min()), 2),
        "spread_pp": round(float(rates.max() - rates.min()), 2),
        "highest_region": str(rates.idxmax()),
        "lowest_region": str(rates.idxmin()),
        "total_regions": int(df["institution_region"].nunique()),
    })

{'highest_region_pct': 23.14, 'lowest_region_pct': 0.12, 'spread_pp': 23.02, 'highest_region': 'Southeast', 'lowest_region': 'U.S. Service Schools', 'total_regions': 10}


In [54]:

pairs = [
        ("institution_name", "institution_control"),
        ("institution_city", "institution_state"),
        ("institution_state", "institution_region"),
    ]

logged = {
        column: np.log1p(
            pd.factorize(df[column])[0].astype(float)
        )
        for column in {column for pair in pairs for column in pair}
    }

print({
        f"{left}~{right}": round(
            float(np.corrcoef(logged[left], logged[right])[0, 1]), 3
        )
        for left, right in pairs
    })

{'institution_name~institution_control': 0.215, 'institution_city~institution_state': nan, 'institution_state~institution_region': nan}


C:\Users\keerthana\AppData\Local\Temp\ipykernel_19732\3535619048.py:8: RuntimeWarning: divide by zero encountered in log1p
  column: np.log1p(
c:\Users\keerthana\Documents\college-set\venv\Lib\site-packages\numpy\lib\_function_base_impl.py:2895: RuntimeWarning: invalid value encountered in subtract
  X -= avg[:, None]


In [55]:

    
print( {
        "total_rows": int(len(df)),
        "missing_institution_names": int(df["institution_name"].isna().sum()),
        "missing_institution_control": int(df["institution_control"].isna().sum()),
        "missing_institution_city": int(df["institution_city"].isna().sum()),
        "missing_institution_state": int(df["institution_state"].isna().sum()),
        "missing_institution_region": int(df["institution_region"].isna().sum()),
        "unique_institutions": int(df["institution_name"].nunique()),
        "unique_controls": int(df["institution_control"].nunique()),
        "unique_cities": int(df["institution_city"].nunique()),
        "unique_states": int(df["institution_state"].nunique()),
        "unique_regions": int(df["institution_region"].nunique()),
        "duplicate_rows": int(df.duplicated().sum()),
        "duplicate_institution_names": int(
            df["institution_name"].duplicated().sum()
        ),
    })

{'total_rows': 39999, 'missing_institution_names': 0, 'missing_institution_control': 0, 'missing_institution_city': 2123, 'missing_institution_state': 2123, 'missing_institution_region': 2123, 'unique_institutions': 1439, 'unique_controls': 4, 'unique_cities': 741, 'unique_states': 55, 'unique_regions': 10, 'duplicate_rows': 0, 'duplicate_institution_names': 38560}


In [58]:

    
data = df if df is not None else load_raw()

print({
        "shape": data.shape,

        "uniqueness": {
            column: int(data[column].nunique())
            for column in data.columns
        },

        "missingness": {
            column: int(data[column].isna().sum())
            for column in data.columns
        },

        "cardinalities": {
            column: int(data[column].nunique())
            for column in data.columns
        },

        "institution_control_distribution":
            institution_distribution(data, "institution_control"),

        "institution_region_distribution":
            institution_distribution(data, "institution_region"),

        "institution_state_distribution":
            institution_distribution(data, "institution_state"),

        "institution_city_distribution":
            institution_distribution(data, "institution_city"),

        
    })

institution_control
Foreign                 2.849201
Private, for-profit    43.293954
Private, nonprofit     29.256428
Public                 24.878388
Name: institution_name, dtype: float64
institution_region
Far West                15.357887
Great Lakes              9.798471
Mid East                13.551077
New England              3.822099
Outlying Areas           2.362752
Plains                   5.837387
Rocky Mountains          3.822099
Southeast               20.500347
Southwest                8.547603
U.S. Service Schools     0.069493
Name: institution_name, dtype: float64
institution_state
AK     0.416956
AL     1.181376
AR     1.528839
AS     0.069493
AZ     2.571230
CA    12.022238
CO     1.806810
CT     0.625434
DC     0.138985
DE     0.138985
FL     3.683113
FM     0.069493
GA     2.223767
HI     0.138985
IA     0.903405
ID     0.764420
IL     2.571230
IN     1.042391
KS     0.972898
KY     1.250869
LA     1.598332
MA     2.084781
MD     1.111883
ME     0.486449
MH     0.